# 06 — App Model Export

This notebook prepares the final XGBoost pipeline for the Streamlit demo app.

The purpose of this notebook is to:

1. Load the cleaned private dataset.
2. Train the final XGBoost model pipeline.
3. Save the trained model locally as `models/xgb_model.pkl`.
4. Create a safe synthetic demo input file for the Streamlit app.

The trained model and private dataset are not uploaded to GitHub.

## Privacy and Deployment Note

The original dataset contains private biomedical data and is not shared publicly.

The trained model file is saved locally under:

`models/xgb_model.pkl`

However, the `models/` folder is ignored by Git to avoid uploading model artifacts unnecessarily.

The Streamlit app can be reproduced by running this notebook locally.

In [1]:
import pandas as pd
import joblib
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

In [2]:
BASE_DIR = Path("..")

DATA_PATH = BASE_DIR / "data" / "processed" / "pd_cleaned.csv"
MODEL_DIR = BASE_DIR / "models"
APP_DIR = BASE_DIR / "app"

MODEL_DIR.mkdir(exist_ok=True)
APP_DIR.mkdir(exist_ok=True)

print("Cleaned data path:", DATA_PATH)
print("Data exists:", DATA_PATH.exists())
print("Model directory:", MODEL_DIR)
print("App directory:", APP_DIR)

(224, 16)
status
1    184
0     40
Name: count, dtype: int64


## Load Cleaned Dataset

The cleaned dataset was produced in the data exploration notebook.

It has already removed:
- identifier columns
- duplicate rows
- constant-value features

The target column is `status`.

In [3]:
df = pd.read_csv(DATA_PATH)

TARGET_COL = "status"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print("Dataset shape:", df.shape)
print("Feature shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

## Train Final XGBoost Pipeline

The final app model uses the same XGBoost configuration used in the modeling notebooks.

The pipeline includes:
- `StandardScaler`
- `XGBClassifier`

This ensures the app applies the same preprocessing steps during inference.

In [4]:
xgb_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

xgb_pipeline.fit(X, y)

print("Final XGBoost pipeline trained successfully.")

Model saved to: ..\models\xgb_model.pkl


## Save Model Locally

The trained model is saved as:

`models/xgb_model.pkl`

This file is used by the Streamlit app.

The model file is intentionally not uploaded to GitHub because the `models/` folder is ignored in `.gitignore`.

In [ ]:
MODEL_PATH = MODEL_DIR / "xgb_model.pkl"

joblib.dump(xgb_pipeline, MODEL_PATH)

print("Model saved to:", MODEL_PATH)
print("Model file exists:", MODEL_PATH.exists())

## Create Safe Demo Input

To avoid exposing real patient-derived samples, the app demo input is created using median feature values from the cleaned dataset.

This produces a synthetic representative sample with the same feature structure required by the model.

The sample input does not include the target column `status`.

In [ ]:
sample_input = X.median().to_frame().T

sample_input_path = APP_DIR / "sample_input.csv"
sample_input.to_csv(sample_input_path, index=False)

print("Sample input saved to:", sample_input_path)
print("Sample input shape:", sample_input.shape)

sample_input

## Export Summary

This notebook generated:

1. A locally saved XGBoost model pipeline:
   - `models/xgb_model.pkl`

2. A safe demo input file:
   - `app/sample_input.csv`

The model artifact is ignored by Git and must be regenerated locally by running this notebook.

The demo input file is safe to upload because it is based on median feature values rather than an individual patient record.